# VEX / vanna — VEX vs IV / range

**Paper:** Vanna / VEX (PDF pp. 6–8) · **NOTES:** [`NOTES.md`](NOTES.md) · **EXP:** [`EXP_REPORT.md`](EXP_REPORT.md)  
**Artifacts:** `out/vex_vanna/`

VEX uses the same PROXY DDOI × BS vanna. Join to HL range/RV for stress context. IV placebo belongs in `robustness`. **Hold**.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

BOOK = Path(os.environ.get("ARES_MICROSTRUCTURE") or (Path.home() / "srv" / "ares-microstructure")) / "research" / "books" / "squeeze_metrics"
OUT = BOOK / "out"
SCRIPTS = BOOK / "scripts"
ROOT = BOOK.parents[2]
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(ROOT))
from _nb_common import (
    load_json, gex_day_table, safe_corr, partial_corr_vs_rv, lead_lag_day, SIGNAL_BOARD,
)
from certified_panel import load_certified, primary_days, spot_l2_days, gex_panel_days, DDOI_LABEL, GEX_PROXY_LABEL
from research.lib import squeeze  # noqa: E402

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

def show_pngs(fig_dir: Path, pattern="*.png"):
    if not fig_dir.is_dir():
        print("no figs dir", fig_dir)
        return
    for p in sorted(fig_dir.glob(pattern)):
        display(Markdown(f"**{p.name}** — `{p.relative_to(BOOK)}`"))
        display(Image(filename=str(p)))

def print_board(rows=None):
    rows = rows or SIGNAL_BOARD
    print(f"{'id':34s} {'gate':6s} note")
    print("-" * 88)
    for i, g, n in rows:
        print(f"{i:34s} {g:6s} {n}")

CERT = load_certified()
PRIMARY = primary_days(CERT)
SPOT = spot_l2_days(CERT)
GEX_DAYS = gex_panel_days(CERT)
print("BOOK", BOOK)
print("certified primary n=", len(PRIMARY), PRIMARY)
print("panel_gex_options n=", len(GEX_DAYS), GEX_DAYS)
print("spot_l2 subpanel n=", len(SPOT), SPOT)
print("DDOI label", DDOI_LABEL, "| GEX proxy", GEX_PROXY_LABEL)
print("trade_synth QUARANTINED — never primary; warehouse OI futures-only unused")


In [ ]:

summary = load_json(OUT / "vex_vanna" / "summary.json")
rows = (summary or {}).get("rows") or []
tbl = gex_day_table(rows, CERT)
print(json.dumps({k: (summary or {}).get(k) for k in ("n_ok","days_ok","vex_mean","corr_vex_hl_range","decision")}, indent=2, default=str))
display(tbl.round(5) if len(tbl) else tbl)


## VEX / GEX+ vs range


In [ ]:

ok = tbl[tbl["ok"] == True].copy() if len(tbl) else tbl
fig, axes = plt.subplots(1, 3, figsize=(12.8, 4.0))
if len(ok):
    axes[0].bar(ok["day"], ok["vex"], color="#3d5a80")
    axes[0].axhline(0, color="k", lw=0.8)
    axes[0].tick_params(axis="x", rotation=70); axes[0].set_title("Day VEX")
    axes[1].scatter(ok["vex"], ok["hl_range"], s=50, c="#ee6c4d")
    axes[1].set_xlabel("VEX"); axes[1].set_ylabel("HL range")
    axes[1].set_title(f"corr={safe_corr(ok['vex'], ok['hl_range'])}")
    axes[2].scatter(ok["gex"], ok["vex"], s=50, c=ok["hl_range"], cmap="magma")
    axes[2].set_xlabel("GEX"); axes[2].set_ylabel("VEX"); axes[2].set_title("GEX vs VEX (color=range)")
else:
    for ax in axes:
        ax.text(0.5, 0.5, "run exp_vex_panel.py", ha="center", transform=ax.transAxes)
plt.tight_layout(); plt.show()
if len(ok):
    print("corr VEX~range", safe_corr(ok["vex"], ok["hl_range"]), "GEX+~range", safe_corr(ok["gex_plus"], ok["hl_range"]))


## Signal board


In [ ]:

print_board([
    ("risk.vex_exposure", "Hold", f"n_ok={(summary or {}).get('n_ok')}; corr_range={(summary or {}).get('corr_vex_hl_range')}"),
    ("risk.squeeze_intensity", "Hold", "GEX+ downstream"),
    ("alpha.tob_cross_arb", "Kill", "never Promote"),
])
